# Pandas: manejar taules de dades de veritat

**Optativa d'Aprenentatge automàtic · DAM/DAW 2n**

A NumPy heu après a manejar arrays: files i columnes de números, indexació,
operacions vectoritzades. Amb Iris ja n'hi havia prou perquè totes les
columnes eren números i cap valor faltava.

Un fitxer de dades de veritat no ve així. Té columnes de text (`Suburb`,
`Type`), columnes numèriques (`Price`, `Rooms`) barrejades a la mateixa
taula, i **valors que falten**. NumPy no distingeix tipus de columna: tot
l'array ha de ser del mateix `dtype`. **Pandas** és la capa que es posa
sobre NumPy per resoldre exactament això: taules amb noms de columna, tipus
diferents per columna, i eines per netejar-les.

Aquest quadern fa servir un dataset real, `melb_data.csv`: 13.580 habitatges
venuts a Melbourne, amb preu, habitacions, superfície i barri. És el mateix
que fareu servir als exercicis.

## Els objectes d'aquest quadern

Aquí tot passa per dos objectes de Pandas: el `DataFrame` (la taula sencera) i la
`Series` (una sola columna). Aquesta taula és la referència que heu de tenir a la
vista mentre feu els exercicis: els noms dels mètodes no s'han d'endevinar, són
aquests.

### `DataFrame` — la taula

És el que torna `pd.read_csv(...)`, `pd.DataFrame({...})`, un filtre (`df[df["Price"] > 1e6]`)
o una selecció de diverses columnes (`df[["Suburb", "Price"]]`).

| Atribut (sense parèntesis) | Què val |
| --- | --- |
| `.shape` | la forma, com a tupla: `(13580, 21)` són 13.580 files i 21 columnes |
| `.columns` | els noms de les columnes. `.columns.tolist()` els dona com a llista |
| `.index` | les etiquetes de les files. Amb un filtre deixa de començar per 0 |
| `.dtypes` | el tipus de cada columna, una per una |
| `.values` | el contingut com a `ndarray` de NumPy, sense noms de columna |
| `.loc[...]` | selecció per **etiqueta** de fila i nom de columna |
| `.iloc[...]` | selecció per **posició** de fila i de columna |

`.loc` i `.iloc` són atributs, no mètodes, i per això porten **claudàtors** i no
parèntesis: `df.loc[0, "Suburb"]`, mai `df.loc(0, "Suburb")`.

| Mètode (amb parèntesis) | Què fa |
| --- | --- |
| `.head(n)` / `.tail(n)` | les primeres / últimes `n` files. Per defecte, 5 |
| `.info()` | files, columnes, tipus i quants valors no nuls té cadascuna |
| `.describe()` | compte, mitjana, desviació, mínim, quartils i màxim de les columnes numèriques |
| `.isna()` | una taula de `True`/`False`: `True` on falta el valor |
| `.dropna(subset=[...])` | elimina les files amb nuls. Sense `subset`, mira totes les columnes |
| `.fillna(valor)` | omple els nuls amb el valor que li doneu |
| `.groupby("columna")` | agrupa les files per cada valor d'aquella columna |
| `.agg(["mean", "median", "count"])` | aplica diverses funcions d'agregació de cop |
| `.sort_values("columna", ascending=False)` | ordena per una columna |
| `.drop(columns=[...])` | torna la taula sense aquelles columnes |
| `.copy()` | una còpia independent. Cal quan voleu afegir columnes a un subconjunt filtrat |
| `.duplicated()` | `True` a les files que repeteixen una d'anterior |
| `.drop_duplicates()` | torna la taula sense les files repetides |
| `.plot(kind="hist")` | dibuixa. `kind` pot ser `"hist"`, `"scatter"`, `"bar"`, `"line"` |

### `Series` — una sola columna

És el que dona `df["Price"]`: una columna amb el seu índex. Té tot el del `DataFrame`
que té sentit per a una sola columna (`.shape`, `.values`, `.mean()`, `.isna()`,
`.fillna()`, `.sort_values()`, `.plot()`...) i a més:

| Mètode | Què fa |
| --- | --- |
| `.value_counts()` | quins valors hi ha i quantes vegades surt cadascun, de més a menys |
| `.unique()` | els valors diferents que hi ha, sense comptar-los |
| `.nunique()` | quants valors diferents hi ha |
| `.map({vell: nou})` | substitueix cada valor segons un diccionari |
| `.astype(tipus)` | converteix la columna a un altre tipus (`int`, `float`, `str`) |

### Atribut o mètode: la distinció que més costa

Un **atribut** és un valor que l'objecte ja porta a dins, i es llegeix **sense
parèntesis**. Un **mètode** és un càlcul que l'objecte sap fer, i s'invoca **amb
parèntesis**.

```python
df.shape      # atribut: (13580, 21)
df.shape()    # TypeError: 'tuple' object is not callable
df.head()     # metode: les 5 primeres files
df.head       # sense parentesis no torna cap fila: imprimeix el metode mateix
```

Confondre-ho no dona un resultat equivocat, dona un `TypeError` o un objecte estrany a
pantalla. El quadern `FO_00_objectes_i_autocompletar.ipynb` ho tracta a fons.

Dues coses que podeu fer sempre, i també a Colab:

- escriure `objecte.` i prémer **TAB**: surt la llista de tot el que aquell objecte té.
- escriure `objecte.metode?` i executar la cel·la: surt la documentació del mètode,
  amb els arguments que accepta.

## 0. Carregar les dades

La cel·la següent carrega `melb_data.csv`. És una mica llarga perquè ha de
funcionar tant en aquest ordinador (on el fitxer ja hi és, a `dades/`) com a
Google Colab (on no hi ha cap fitxer local): si no el troba, el baixa del
repositori del mòdul a GitHub. Si tampoc hi ha
connexió, genera un substitut amb la mateixa forma perquè el quadern no
peti, encara que els números concrets del text ja no hi quadrin.

No cal entendre-la línia a línia: és fontaneria, no el tema d'avui. El tema
d'avui comença al punt 1, amb `df` ja carregat.

In [ ]:
import os
import urllib.request

import numpy as np
import pandas as pd

RUTA_LOCAL = "dades/melb_data.csv"
URL_PUBLICA = (
    "https://raw.githubusercontent.com/ZenidX/Machine-Learning/main/"
    "Machine%20Learning/01_fonaments/dades/melb_data.csv"
)


def fabrica_substitut(n=13580, llavor=42):
    """Substitut sintètic amb la mateixa forma que melb_data.csv.

    Nomes s'utilitza si no hi ha fitxer local ni connexio: cap dels numeros
    del text d'aquest quadern hi encaixara, pero el codi seguira funcionant.
    """
    rng = np.random.default_rng(llavor)
    barris = [f"Barri_{i}" for i in range(1, 21)]
    tipus = rng.choice(["h", "u", "t"], size=n, p=[0.7, 0.22, 0.08])
    habitacions = rng.integers(1, 6, size=n)
    preu_base = 300_000 + habitacions * 250_000
    preu = preu_base * rng.lognormal(mean=0, sigma=0.3, size=n)
    landsize = rng.gamma(shape=2.0, scale=250, size=n)
    building_area = np.where(
        rng.random(n) < 0.475, np.nan, rng.gamma(shape=3.0, scale=45, size=n)
    )
    return pd.DataFrame({
        "Suburb": rng.choice(barris, size=n),
        "Rooms": habitacions,
        "Type": tipus,
        "Price": preu.round(-3),
        "Distance": rng.gamma(shape=2.0, scale=5, size=n).round(1),
        "Bathroom": rng.integers(1, 4, size=n),
        "Car": rng.integers(0, 4, size=n).astype(float),
        "Landsize": landsize.round(0),
        "BuildingArea": building_area,
        "Regionname": rng.choice(
            ["Northern Metropolitan", "Southern Metropolitan",
             "Western Metropolitan", "Eastern Metropolitan"], size=n
        ),
    })


if os.path.exists(RUTA_LOCAL):
    df = pd.read_csv(RUTA_LOCAL)
    origen = f"fitxer local ({RUTA_LOCAL})"
else:
    try:
        os.makedirs(os.path.dirname(RUTA_LOCAL) or ".", exist_ok=True)
        urllib.request.urlretrieve(URL_PUBLICA, RUTA_LOCAL)
        df = pd.read_csv(RUTA_LOCAL)
        origen = "descarregat de GitHub (funcionem a Colab)"
    except Exception as e:
        df = fabrica_substitut()
        origen = f"substitut sintètic (sense fitxer local ni connexió: {e})"

print(f"Dades carregades des de: {origen}")
print(f"Files: {len(df)}, columnes: {df.shape[1]}")

## 1. Què afegeix Pandas sobre NumPy

Agafem 5 habitatges i només tres de les seves columnes, i mirem-los primer
com a array de NumPy i després com a taula de Pandas.

In [ ]:
mostra = df[["Suburb", "Rooms", "Price"]].head(5)

array = mostra.to_numpy()
print("Com a array de NumPy:")
print(array)
print("dtype de l'array:", array.dtype)

Tot l'array és `dtype=object`: NumPy no pot tenir una columna de text
(`Suburb`) i dues de números (`Rooms`, `Price`) amb tipus diferents, així
que ho converteix tot al tipus més genèric. `Rooms` i `Price` ja no són
números de veritat, són text disfressat de número. I l'array no sap com es
diu cada columna: `array[:, 2]` és el preu només perquè ho recordeu
vosaltres, no perquè l'array ho digui.

In [ ]:
print("Com a DataFrame de Pandas:")
mostra

El DataFrame recorda els noms de columna (`Suburb`, `Rooms`, `Price`) i el
tipus de cadascuna per separat. `mostra["Price"]` és sempre el preu, i
segueix sent un número amb el qual es pot calcular. Aquesta és tota la
diferència: **un array és una graella de números; un DataFrame és una taula
amb noms i tipus, com un full de càlcul.**

## 2. Series i DataFrame

Un **DataFrame** és la taula sencera. Una **Series** és una sola columna (o
una sola fila). Es poden crear des de zero amb un diccionari, sense llegir
cap fitxer, per veure'n l'estructura amb dades petites i conegudes.

In [ ]:
alumnes = pd.DataFrame({
    "nom": ["Laia", "Marc", "Judit", "Pol"],
    "grup": ["DAM1A", "DAW1B", "DAM1A", "DAW1B"],
    "nota": [7.5, 5.0, 9.0, 6.5],
})
alumnes

In [ ]:
notes = alumnes["nota"]     # una columna sola: aixo es una Series
print(type(alumnes))
print(type(notes))
notes

Fixeu-vos en la columna de l'esquerra, sense nom: és l'**índex**. Per
defecte és 0, 1, 2, 3..., però no és una columna de dades, és l'etiqueta de
cada fila. Servirà d'aviat per a una confusió molt concreta (punt 4).

Les eines bàsiques per fer-se una idea ràpida d'un DataFrame, totes en una
cel·la:

In [ ]:
print("shape:", alumnes.shape)          # (files, columnes)
print("columns:", alumnes.columns.tolist())
print("dtypes:")
print(alumnes.dtypes)
print()
alumnes.info()

In [ ]:
alumnes.describe()

`describe()` només fa estadístiques de les columnes **numèriques** (aquí,
`nota`): compta, mitjana, desviació, mínim, quartils i màxim. Les columnes
de text (`nom`, `grup`) les ignora. `head(n)` i `tail(n)` ensenyen les
primeres o últimes `n` files; per defecte, 5.

## 3. Carregar dades de veritat

Ja hem carregat `melb_data.csv` a la cel·la del punt 0, amb exactament la
instrucció que fareu servir sempre que tingueu un fitxer local:

```python
df = pd.read_csv("dades/melb_data.csv")
```

Abans de fer-hi res, **es mira**, sense excepcions. Quantes files i
columnes té, quines columnes hi ha, com són els primers registres.

In [ ]:
print("Files:", df.shape[0], " Columnes:", df.shape[1])
print()
print(df.columns.tolist())

In [ ]:
df.head()

Cada fila és un habitatge venut: barri, adreça, habitacions, tipus, preu,
mètode de venda, distància al centre, superfície... 21 columnes. Per a les
columnes de text, `value_counts()` diu quins valors hi ha i amb quina
freqüència, cosa que `describe()` no fa perquè no són números.

In [ ]:
df["Type"].value_counts()

`Type` només té tres valors: `h` (house, casa), `u` (unit, pis o duplex) i
`t` (townhouse, adossat). La immensa majoria són cases. Proveu-ho també amb
`Regionname`: hi ha 8 regions, i les tres metropolitanes del sud, nord i
oest concentren la majoria dels habitatges.

In [ ]:
df["Regionname"].value_counts()

## 4. Seleccionar

### Una columna i diverses

Una sola columna amb claudàtors simples i el nom (dona una Series);
diverses columnes amb una **llista** de noms dins els claudàtors (dona un
DataFrame, encara que sigui d'una sola columna).

In [ ]:
preus = df["Price"]                      # Series
subset = df[["Suburb", "Rooms", "Price"]]  # DataFrame
print(type(preus), type(subset))
subset.head(3)

### `loc` contra `iloc`: la confusió clàssica

- **`loc`** selecciona per **etiqueta**: el nom de l'índex i el nom de la
  columna.
- **`iloc`** selecciona per **posició**: "la fila 0", "la columna 2",
  independentment de com es diguin.

Amb l'índex per defecte (0, 1, 2...) semblen el mateix, i per això
enganyen:

In [ ]:
print(df.loc[0, "Suburb"])   # etiqueta de fila 0, columna "Suburb"
print(df.iloc[0, 0])         # posicio 0, posicio de columna 0
print(df.loc[0, "Suburb"] == df.iloc[0, 0])

Coincideixen perquè encara ningú ha tocat l'índex. Però filtreu el
DataFrame i l'índex es queda amb els números originals de les files que han
sobreviscut al filtre; deixa de començar per 0. Aquí és on `loc` i `iloc`
es separen de veritat.

In [ ]:
richmond = df[df["Suburb"] == "Richmond"]
print("Primers índexs de richmond:", richmond.index[:5].tolist())

print(richmond.iloc[0]["Suburb"])   # posicio 0 del subset: sempre funciona
try:
    print(richmond.loc[0])          # etiqueta 0: no existeix en aquest subset
except KeyError as e:
    print("KeyError:", e)

`richmond.iloc[0]` és "la primera fila que hi ha", sigui quina sigui la
seva etiqueta. `richmond.loc[0]` busca la fila **etiquetada** `0`, i com que
la fila 0 original (`Abbotsford`) no és de Richmond, no hi és: `KeyError`.

**Regla pràctica:** si voleu dir "la fila número tal en l'ordre en què la
veig", és `iloc`. Si voleu dir "la fila que porta aquesta etiqueta concreta"
(sovint perquè ja sabeu quin índex busqueu), és `loc`. En cas de dubte amb
un DataFrame filtrat, gairebé sempre voleu `iloc`.

### Filtrar amb condicions

`df[condicio]` es queda només amb les files on la condició és certa. La
condició és una Series de `True`/`False`, de la mateixa mida que `df`.

In [ ]:
cars = df[df["Price"] > 1_000_000]
print(f"Habitatges de més d'1M: {len(cars)} de {len(df)}")

### Combinar condicions: `&`, `|`, i per què calen els parèntesis

Per combinar condicions **no** es fa servir `and`/`or` de Python (petaria):
es fa servir `&` (i) i `|` (o). I **cada condició ha d'anar entre
parèntesis**, perquè `&` i `|` tenen més prioritat que `==` i `>` en Python:
sense parèntesis, Python intenta combinar-los abans de comparar, i peta o
dona un resultat que no té sentit.

In [ ]:
# amb parentesis: correcte
tres_habitacions_cares = df[(df["Rooms"] == 3) & (df["Price"] > 1_000_000)]
print(f"3 habitacions i > 1M: {len(tres_habitacions_cares)}")

try:
    df[df["Rooms"] == 3 & df["Price"] > 1_000_000]   # sense parentesis
except TypeError as e:
    print("TypeError:", e)

Sense parèntesis, Python avalua `3 & df["Price"]` abans que la comparació,
que no vol dir res per a aquestes dades, i peta. La regla és senzilla:
**cada comparació individual, entre parèntesis**, sempre que combineu amb
`&` o `|`.

## 5. Columnes noves i transformacions

### Una columna calculada

Es crea assignant a un nom de columna nou; l'operació es fa fila a fila,
sense bucle.

In [ ]:
df["preu_per_habitacio"] = df["Price"] / df["Rooms"]
df[["Suburb", "Rooms", "Price", "preu_per_habitacio"]].head(3)

### `apply`: una funció pròpia, fila a fila

`apply` passa cada valor de la columna per una funció que escriviu
vosaltres. Útil quan la transformació no és una simple operació aritmètica.

In [ ]:
def banda_preu(preu):
    if preu < 500_000:
        return "barat"
    elif preu < 1_000_000:
        return "mitjà"
    else:
        return "car"


df["banda_preu"] = df["Price"].apply(banda_preu)
df["banda_preu"].value_counts()

### `map`: recodificar valors coneguts

Quan la transformació és "substitueix aquest valor per aquest altre", i ja
sabeu tots els valors possibles, `map` amb un diccionari és més directe que
`apply` amb una funció `if/elif`.

In [ ]:
df["tipus_llarg"] = df["Type"].map({"h": "casa", "u": "unitat o duplex", "t": "adossat"})
df["tipus_llarg"].value_counts()

## 6. Valors que falten

Amb `melb_data.csv` això no és un exercici teòric: hi ha columnes amb molts
nuls de veritat. Primer, l'inventari.

In [ ]:
nuls = df.isna().sum()
nuls[nuls > 0].sort_values(ascending=False)

In [ ]:
percentatge = (df.isna().mean() * 100).round(1)
percentatge[percentatge > 0].sort_values(ascending=False)

`BuildingArea` falta en gairebé la meitat de files, `YearBuilt` en quatre de
cada deu, `CouncilArea` en una de cada deu, i `Car` gairebé mai. No totes
les columnes amb nuls es tracten igual.

### `dropna`: què costa en files

`dropna()` sense arguments elimina **qualsevol fila amb algun nul en
qualsevol columna**. Amb `BuildingArea` faltant a gairebé la meitat de
files, el cost és brutal:

In [ ]:
sencer = df.dropna()
print(f"Files abans: {len(df)}")
print(f"Files despres de dropna() sencer: {len(sencer)}")
print(f"Percentatge perdut: {100 * (1 - len(sencer) / len(df)):.1f} %")

Es perd més de la meitat del dataset per culpa, sobretot, d'una sola
columna. `dropna(subset=[...])` permet limitar-ho a les columnes que
realment us importen en aquell moment, en lloc de a totes.

### `fillna`: omplir en lloc d'eliminar

Per a una columna amb pocs nuls, com `Car` (nombre de places d'aparcament),
té més sentit omplir-los que perdre la fila sencera. Un valor habitual és
la **mediana** (menys sensible als extrems que la mitjana):

In [ ]:
print("Nuls a Car abans:", df["Car"].isna().sum())

mediana_car = df["Car"].median()
df["Car"] = df["Car"].fillna(mediana_car)

print("Mediana feta servir:", mediana_car)
print("Nuls a Car despres:", df["Car"].isna().sum())

**Per què mediana i no mitjana**, aquí i en general: `Price` té mitjana al
voltant d'1.075.000 però mediana al voltant de 903.000. La diferència ve
d'un grapat d'habitatges molt cars que estiren la mitjana cap amunt; la
mediana no els fa cas. Per a `Car`, que va de 0 a 8 amb la immensa majoria
en 1 o 2, mitjana i mediana gairebé coincideixen, però l'hàbit de fer servir
la mediana per defecte evita sorpreses quan la columna sí que estigui
esbiaixada.

## 7. Agrupar

`groupby` respon preguntes del tipus "una xifra per cada valor d'aquesta
columna". Preu mitjà per nombre d'habitacions:

In [ ]:
df.groupby("Rooms")["Price"].mean().round(0)

Es pot agrupar per **més d'una columna** alhora, passant una llista. Preu
mitjà per regió i per nombre d'habitacions (només les primeres files, la
taula sencera és llarga):

In [ ]:
df.groupby(["Regionname", "Rooms"])["Price"].mean().round(0).head(8)

### `agg`: diverses funcions alhora

En lloc d'una sola funció (`mean`), `agg` en pot aplicar diverses de cop, en
forma de llista.

In [ ]:
df.groupby("Regionname")["Price"].agg(["mean", "median", "count"]).round(0)

Es veu de seguida per què la mediana importa: a totes les regions, la
mitjana surt per sobre de la mediana, senyal que uns quants habitatges molt
cars estiren la mitjana amunt a totes bandes.

### `pivot_table`: la mateixa idea, en format taula

`groupby` amb dues columnes dona una llista llarga (una fila per
combinació). `pivot_table` posa una de les dues columnes com a **capçalera**,
que sovint es llegeix més fàcil: preu mitjà per regió (files) i nombre
d'habitacions (columnes).

In [ ]:
taula = df.pivot_table(values="Price", index="Regionname", columns="Rooms", aggfunc="mean")
taula.round(0).iloc[:, :5]

És exactament la mateixa informació que el `groupby` d'abans, reorganitzada
perquè es puguin comparar files i columnes d'un cop d'ull. Es veu, per
exemple, que a `Southern Metropolitan` el preu puja molt més ràpid amb cada
habitació que a `Western Victoria`.

## 8. Ordenar i comptar

`sort_values` ordena tot el DataFrame per una columna. Els 5 habitatges més
cars:

In [ ]:
df.sort_values("Price", ascending=False)[["Suburb", "Rooms", "Price"]].head(5)

`nlargest`/`nsmallest` fan el mateix però només calculen el que cal, sense
ordenar tota la taula: més ràpid quan el dataset és gran i només voleu els
extrems.

In [ ]:
print("Els 3 mes cars:")
print(df.nlargest(3, "Price")[["Suburb", "Price"]])
print()
print("Els 3 mes barats:")
print(df.nsmallest(3, "Price")[["Suburb", "Price"]])

## 9. Gràfics ràpids

Un DataFrame o una Series es poden dibuixar directament amb `.plot()`, sense
tocar matplotlib a fons (encara). Distribució de preus:

In [ ]:
import matplotlib.pyplot as plt

df["Price"].plot(kind="hist", bins=50, figsize=(8, 4), title="Distribucio de preus")
plt.xlabel("Preu")
plt.show()

Dispersió entre dues columnes numèriques, amb `kind="scatter"`: preu segons
distància al centre.

In [ ]:
df.plot(kind="scatter", x="Distance", y="Price", alpha=0.2, figsize=(8, 5),
        title="Preu segons distancia al centre")
plt.show()

I un gràfic de barres a partir d'un `value_counts()`, per a columnes de
text:

In [ ]:
df["Type"].value_counts().plot(kind="bar", figsize=(6, 4), title="Habitatges per tipus")
plt.ylabel("Nombre d'habitatges")
plt.show()

## 10. Del DataFrame al model

Tot el curs gira al voltant d'aquest pas: scikit-learn no accepta un
DataFrame amb columnes de text i valors que falten. Vol una `X` numèrica
(les característiques) i una `y` (el que es vol predir), totes dues sense
nuls.

Els passos, sempre:

1. Triar només les columnes **numèriques** que us interessen (les de text,
   com `Suburb` o `Regionname`, o es descarten o es recodifiquen a números;
   avui les descartem).
2. Eliminar les files amb nuls en aquestes columnes (`dropna`), o omplir-los
   abans (punt 6).
3. Passar-ho a arrays amb `.values` (o `.to_numpy()`, és equivalent).
4. `train_test_split`, entrenar, avaluar. Exactament com als quaderns de
   teoria.

In [ ]:
columnes_model = ["Rooms", "Distance", "Bathroom", "Car", "Landsize", "BuildingArea"]

dades_model = df[columnes_model + ["Price"]].dropna()
print(f"Files utilitzables: {len(dades_model)} de {len(df)}")

X = dades_model[columnes_model].values
y = dades_model["Price"].values
print("Forma de X:", X.shape, " Forma de y:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X_entrena, X_examen, y_entrena, y_examen = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = LinearRegression()
model.fit(X_entrena, y_entrena)

r2 = model.score(X_examen, y_examen)
print(f"R2 a l'examen: {r2:.3f}")

Un R² de `0,40` vol dir que el model explica un 40 % de la variació dels
preus amb només sis columnes numèriques senzilles: ni és perfecte ni és
inútil. Li falta la informació més determinant del preu real (el barri
concret), que ara mateix és text i queda fora. Recodificar columnes de text
a números (`pd.get_dummies`, one-hot encoding) és el pas natural següent,
però queda fora d'aquest quadern.

**Aquest és el tancament del bloc de fonaments.** Heu vist arbres, boscos i
regressions amb Iris ja preparat; ara heu vist d'on surt aquest `X, y` quan
les dades arriben brutes, amb text i amb forats. A partir d'aquí, cap
dataset de veritat us hauria de fer por.

## 11. Pràctica

Tots els exercicis fan servir `df`, tal com ha quedat després de tot el
quadern (amb `Car` ja omplert i les columnes `preu_per_habitacio`,
`banda_preu` i `tipus_llarg` afegides).

### Exercici 1 — Habitatges per barri

Quants habitatges hi ha a cada barri (`Suburb`)? Troba els 5 barris amb més
habitatges venuts.

**Com saps que ho has fet bé:** pots donar el nom exacte del barri amb més
vendes i el nombre exacte d'habitatges que té, sense tornar a mirar el
codi.

In [ ]:
# Exercici 1

# compta els habitatges per barri amb .value_counts() sobre la columna Suburb
# (ja ve ordenat de més a menys)

# queda't amb els 5 primers amb .head(5)

### Exercici 2 — Preu mitjà segons el nombre d'habitacions

Calcula el preu mitjà per cada valor de `Rooms`. La tendència és creixent,
però no del tot: hi ha algun valor de `Rooms` on el preu mitjà **baixa**
respecte de l'anterior. Troba quin.

**Com saps que ho has fet bé:** pots assenyalar el valor concret de `Rooms`
on la tendència es trenca, i una hipòtesi curta (una frase) de per què hi
pot haver poques dades justament allà.

In [ ]:
# Exercici 2

# .groupby("Rooms") sobre la columna "Price", i .mean()
# arrodoneix-ho amb .round(0) per llegir-ho millor

# mira la taula: en quin punt baixa en lloc de pujar?
# per saber si allà hi ha poques dades, repeteix-ho amb .agg(["mean", "count"])

### Exercici 3 — La columna més buida

Ja hem vist a la teoria que `BuildingArea` és la que té més nuls, però
comprova-ho tu amb codi, sense donar-ho per fet, i calcula exactament quin
percentatge de files hi falta aquesta columna.

**Com saps que ho has fet bé:** el número que calculis ha de coincidir amb
el que hem vist al punt 6 (arrodonit a un decimal).

In [ ]:
# Exercici 3

# .isna().sum() dona els nuls de cada columna; ordena'ls amb
# .sort_values(ascending=False) i queda't amb el primer amb .head(1)
# calcula el percentatge amb .isna().mean() * 100, i arrodoneix amb .round(1)

### Exercici 4 — Preu per metre quadrat

Crea una columna `preu_m2` amb `Price / Landsize`. **Compte**: hi ha
habitatges amb `Landsize` igual a 0, que farien una divisió per zero (o un
`preu_m2` absurdament alt si el numerador no és zero). Filtra'ls abans de
calcular res. Amb la resta, troba els 5 barris amb el `preu_m2` mitjà més
alt.

**Com saps que ho has fet bé:** el nombre de files que fas servir per
calcular-ho ha de ser menor que `len(df)`, i pots dir per quants habitatges
exactament ha disminuït.

In [ ]:
# Exercici 4

# filtra df on Landsize > 0, i fes .copy() del resultat
# (sense la còpia, afegir-hi una columna avisa amb un SettingWithCopyWarning)
# crea la columna preu_m2 = Price / Landsize sobre aquest subconjunt filtrat
# agrupa amb .groupby("Suburb"), fes .mean() de preu_m2,
# ordena amb .sort_values(ascending=False) i queda't amb els 5 primers amb .head(5)

### Exercici 5 — Filtrar per dos criteris

Troba tots els habitatges de tipus casa (`Type == "h"`) a la regió
`"Southern Metropolitan"` amb menys de 3 habitacions. Quants n'hi ha? Quin
és el seu preu mitjà, comparat amb el preu mitjà de totes les cases de la
mateixa regió (sense el límit d'habitacions)?

**Com saps que ho has fet bé:** el preu mitjà del grup filtrat (menys
habitacions) hauria de sortir **per sota** del preu mitjà general de la
regió. Si et surt per sobre, revisa la condició.

In [ ]:
# Exercici 5

# filtra per Type == "h", Regionname == "Southern Metropolitan" i Rooms < 3
# (recorda els parentesis a cada condicio combinada amb &)
# compta les files del resultat amb len()
# compara el preu mitjà d'aquest grup amb .mean() sobre la seva columna "Price",
# contra el mateix càlcul sobre les cases de la regio sense el límit d'habitacions

### Exercici 6 — Quin tipus d'habitatge és més car

Calcula el preu mitjà per cada valor de `Type` (`h`, `u`, `t`) i ordena'ls
de més car a més barat amb `sort_values`. Fes-ho també amb `tipus_llarg`
(la columna que vau crear amb `map` al punt 5) i comprova que l'ordre és el
mateix.

**Com saps que ho has fet bé:** l'ordre ha de sortir idèntic amb `Type` i
amb `tipus_llarg`: és la mateixa informació amb noms diferents, `map` no
canvia les dades, només com es lleigeixen.

In [ ]:
# Exercici 6

# preu mitjà amb .groupby("Type") sobre "Price" i .mean(),
# ordenat de mes car a mes barat amb .sort_values(ascending=False)
# repeteix-ho agrupant per tipus_llarg i compara l'ordre

### Exercici 7 — Un model sense la columna més buida

Al punt 10 hem entrenat un `LinearRegression` amb `BuildingArea` inclosa, i
això ens ha costat gairebé la meitat de les files (`dropna` de sis
columnes). Entrena el mateix model però **sense** `BuildingArea` a
`columnes_model`, i compara dues coses amb el resultat del punt 10: quantes
files es fan servir ara, i quin R² surt.

**Com saps que ho has fet bé:** hauries de conservar moltes més files que
al punt 10 (menys columnes amb nuls, menys files eliminades). Si l'R² surt
millor, pitjor o pràcticament igual, digues quina de les dues coses creus
que pesa més: tenir més files, o tenir una columna menys que sí que es
relaciona amb el preu.

In [ ]:
# Exercici 7

# repeteix el punt 10 amb columnes_model sense "BuildingArea":
# .dropna() sobre les columnes triades, .values per treure X i y,
# train_test_split, LinearRegression().fit() i .score()
# compara nombre de files utilitzades i R2 amb el resultat original

## Resum

- Un **array** de NumPy és una graella de números sense noms de columna ni
  tipus mixtos. Un **DataFrame** de Pandas sí que en té: és la peça que fa
  falta per manejar un fitxer de dades real.
- `read_csv` carrega un fitxer; `head`, `shape`, `dtypes`, `info`,
  `describe` i `value_counts` són els primers cinc minuts amb qualsevol
  dataset nou, sempre.
- **`loc` és per etiqueta, `iloc` és per posició.** Coincideixen amb l'índex
  per defecte i es separen en el mateix moment en què filtreu el
  DataFrame: aquí és on la confusió sol aparèixer.
- Filtrar amb `&` i `|` exigeix parèntesis a cada condició; sense ells,
  Python els combina abans de comparar i peta.
- `isna().sum()` fa l'inventari de nuls; `dropna()` els elimina (i pot
  costar més de la meitat de les files); `fillna()` els omple, sovint amb
  la mediana.
- `groupby`, `agg` i `pivot_table` responen "una xifra per cada categoria",
  amb una, dues o més columnes alhora.
- El pas del DataFrame a `X, y` de scikit-learn és sempre: tria columnes
  numèriques, treu els nuls, `.values`, `train_test_split`.

### I ara què

Amb NumPy i Pandas ja teniu les dues eines que us faltaven per fer sols el
que heu vist a la teoria: carregar un fitxer real, netejar-lo una mica, i
entrenar-hi un model. El bloc següent és tot pràctica, amb datasets nous
que no heu vist abans.